# 从 20 日滚动均线学习导数

**本教案的目标**：理解日线上的“导数”是什么、为什么它没有未来函数，以及如何在任意一个收盘日手算出来。

> 配套材料：[`rolling_mean_derivative_blackboard.nb`](./rolling_mean_derivative_blackboard.nb) 用于课堂上的 Wolfram 演示和交互；本 `.ipynb` 用于讲解、阅读和练习。

In [1]:
# 可选：在经典 Jupyter Notebook/Lab 中运行此格，让中文正文优先使用楷体。
from IPython.display import HTML, display
display(HTML('''
<style>
.rendered_html { font-family: KaiTi, STKaiti, KaiTi_GB2312, serif; font-size: 16px; line-height: 1.75; }
.rendered_html .MathJax { font-family: Times New Roman, serif; }
</style>
'''))

## 0. 先区分两份材料

| 文件 | 课堂角色 |
| --- | --- |
| 本 `.ipynb` | 教案：解释概念、推导、讨论误区、布置练习。 |
| `.nb` 黑板 | 演示：现场运行 Wolfram 代码、替换价格、观察结果。 |

今天先把数学关系想清楚，再把它写成代码。

## 1. 问题：均线图上的“导数”是什么意思？

收盘价和移动均线都只在交易日有一个点，而不是每个实数时刻都有定义的连续曲线。因此，图上最自然的“导数”不是微积分中的极限，而是**相邻交易日的变化率**。

记第 $t$ 个交易日的 20 日均线为 $M_t$。我们定义当天的离散导数：

$$D_t=M_t-M_{t-1}$$

单位是“价格／交易日”。它回答的是：**均线从昨天收盘到今天收盘，移动了多少？**

## 2. 20 日均线的定义

令 $P_t$ 是第 $t$ 个交易日的收盘价。这里使用的是向后看的 20 日简单移动平均（trailing SMA）：

$$M_t=\frac{1}{20}\sum_{k=0}^{19}P_{t-k}=\frac{P_t+P_{t-1}+\cdots+P_{t-19}}{20}$$

窗口是 $[t-19,t]$。在 $t$ 日收盘后，其中每一个价格都已经发生；所以 $M_t$ 没有未来函数。

## 3. 推导：19 项为什么会消失？

把今天和昨天的均线分别写出来：

$$M_t=\frac{P_t+P_{t-1}+\cdots+P_{t-19}}{20}$$

$$M_{t-1}=\frac{P_{t-1}+P_{t-2}+\cdots+P_{t-20}}{20}$$

相减时，$P_{t-1}$ 到 $P_{t-19}$ 同时存在于两边，全部抵消：

$$\begin{aligned}
D_t&=M_t-M_{t-1}\\
&=\frac{P_t-P_{t-20}}{20}
\end{aligned}$$

这就是要记住的结果：

$$\boxed{D_t=\frac{P_t-P_{t-20}}{20}}$$

## 4. 直觉：窗口每次向前滚动一天

从昨天到今天，20 日窗口只做了两件事：

1. 新加入今天的价格 $P_t$；
2. 移除 20 个交易日前的价格 $P_{t-20}$。

所以均线的变化只取决于“进来的数”减去“出去的数”，再均摊到 20 天。

- 若 $P_t>P_{t-20}$，则 $D_t>0$：均线上行。
- 若 $P_t<P_{t-20}$，则 $D_t<0$：均线下行。
- 若两者相等，均线当天不动。

## 5. 为什么它没有未来函数？

在 $t$ 日收盘后，公式只需要 $P_t$ 与 $P_{t-20}$：

$$D_t=\frac{P_t-P_{t-20}}{20}$$

它不包含 $P_{t+1}$、$M_{t+1}$ 或任何未来数据。因此可用于收盘后的回测和信号计算。

**重要反例**：居中滚动均线（例如把窗口放在 $t$ 的两边）会引用未来价格。即使画出来很好看，也不应用作当日可交易的信号。

## 6. 例题

假设今天收盘价为 $P_t=31.80$，20 个交易日前收盘价为 $P_{t-20}=29.40$。

$$D_t=\frac{31.80-29.40}{20}=0.12$$

解释：今天的 20 日均线比昨天高 $0.12$ 个价格单位。它不是价格当天涨了 $0.12$，而是均线每天的位移为 $0.12$。

## 7. 数据长度与日期对齐

- 有 20 个价格后，才有第一个 20 日均线。
- 有 21 个价格后，才有第一个相邻均线差 $D_t$。
- 若 `prices` 按时间从早到晚排列，直接公式在索引 $i=21$ 开始计算。

务必把结果标记到**后一日**：$D_t=M_t-M_{t-1}$ 属于 $t$ 日，而不是 $t-1$ 日。

## 8. Wolfram 黑板上要演示的两种等价算法

在 `.nb` 中可以比较：

```wl
rolling20 = MovingAverage[prices, 20];
derivativeDaily = Differences[rolling20];

derivativeDirect = Table[(prices[[i]] - prices[[i - 20]])/20,
  {i, 21, Length[prices]}];

derivativeDaily == derivativeDirect
```

结果应为 `True`：一种是“先算均线，再和昨天相减”；另一种是我们刚刚手推出来的快捷公式。

## 9. 课后自测

1. 为什么 $D_t$ 不需要知道 $P_{t-1}$ 到 $P_{t-19}$ 的具体数值？
2. 若 $P_t=18$、$P_{t-20}=22$，求 $D_t$ 并解释方向。
3. 为什么 centered rolling mean 不满足本课的无未来函数要求？
4. 画折线时，在交易日的折点是否一定存在唯一的普通导数？为什么？

### 答案要点

1. 两个相邻窗口的 19 个共同项相减为零。
2. $D_t=(18-22)/20=-0.2$，均线向下。
3. 它会引用尚未发生的价格。
4. 不一定；折点的左、右斜率可能不同。我们使用的是离散差分，而非强行定义连续导数。